# Урок 11 - Протокол агент-до-агента (A2A)


## Налаштування


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Що таке протокол A2A?

The **Agent-to-Agent (A2A) protocol** — це відкритий стандарт, який дозволяє агентам ШІ спілкуватися, виявляти один одного та співпрацювати — навіть якщо вони побудовані на різних фреймворках або розміщені на різних сервісах.

Ключові поняття:

- **Discovery** – Агенти публікують *Agent Card*, яка описує їхні можливості, що полегшує іншим агентам (або оркестраторам) знайти відповідного спеціаліста для завдання.
- **Message Passing** – Агенти обмінюються структурованими повідомленнями через спільний протокол, тож запит від одного агента може бути зрозумілий і виконаний іншим незалежно від його внутрішньої реалізації.
- **Task Lifecycle** – A2A визначає стани такі як *submitted*, *working*, *completed* та *failed*, надаючи оркестратору повну видимість того, як просувається делеговане завдання.

У цьому уроці ми імітуємо співпрацю в стилі A2A, підключивши три спеціалізовані туристичні агенти до робочого процесу, де кожен агент вносить свою експертизу і передає результати наступному.


## Створення спеціалізованих туристичних агентів


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Співпраця кількох агентів через робочий процес

Ми з'єднуємо три агенти в послідовний робочий процес, який відображає A2A обмін повідомленнями:

1. **CurrencyExchangeAgent** отримує запит користувача та надає рекомендації щодо валюти.
2. **ActivityPlannerAgent** отримує збагачений контекст і додає рекомендації щодо активностей.
3. **TravelManagerAgent** синтезує обидва вхідні дані в підсумковий бриф подорожі.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Розуміння A2A у виробництві

У виробничому середовищі протокол A2A відкриває потужні сценарії міжсервісної взаємодії:

| Capability | Description |
|---|---|
| **Інтероперабельність між фреймворками** | Агент, створений з одного фреймворку, може делегувати завдання агенту, створеному з будь-якого іншого фреймворку, сумісного з A2A, що забезпечує справжню міжорганізаційну взаємодію. |
| **Service boundaries** | Агенти можуть жити в окремих мікросервісах, регіонах хмари або навіть у різних організаціях, при цьому безперешкодно співпрацюючи. |
| **Динамічне виявлення** | Оркестратор може запитувати реєстр Agent Card під час виконання, щоб знайти найкраще підходящого спеціаліста для певного підзавдання. |
| **Потокова передача & push-повідомлення** | A2A підтримує Server-Sent Events (SSE) для оновлень ходу в реальному часі та push-повідомлення для довготривалих завдань. |

The workflow we built above is a simplified, in-process version of this pattern. In a real
deployment each agent would expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## Підсумок

У цьому уроці ви дізналися:

1. **Що таке протокол A2A** — відкритий стандарт для взаємного виявлення агентів, обміну повідомленнями,
   та управління завданнями.
2. **Як створити спеціалізованих агентів** — агент обміну валют, агент-планувальник активностей,
   і оркестратор Travel Manager.
3. **Як підключити агентів до робочого процесу** — використовуючи `WorkflowBuilder` для моделювання послідовної
   передачі повідомлень між агентами.
4. **Як A2A працює у виробничому середовищі** — дозволяючи співпрацю між різними фреймворками та сервісами
   з динамічним виявленням і потоковими оновленнями.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Відмова від відповідальності:
Цей документ було перекладено з використанням сервісу перекладу на основі ШІ Co-op Translator (https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, просимо врахувати, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критичної інформації рекомендується скористатися професійним перекладом, виконаним людиною. Ми не несемо відповідальності за будь-які непорозуміння чи неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
